# Llama 3.1 8B runtime smoke (not an experiment)

This notebook checks the exact report-aligned checkpoint with the pinned NF4 + LoRA stack and one synthetic optimizer step. It reads no research corpus, creates no research result, and does not start SFT/unlearning.

Before running: select GPU T4 x2 and turn Internet on in Notebook Settings. Under Add-ons → Secrets, enable the existing `HF_TOKEN` secret. The notebook is safe-by-default: set `RUN_SMOKE = True` in the code cell only after those settings are ready. The token is never printed.

The Hugging Face account must have access to `meta-llama/Llama-3.1-8B-Instruct`. The cell records only the resolved commit hash, package versions, GPU information, pass/fail, and synthetic-step metrics.


In [ ]:
# Change this to True only after enabling GPU, Internet, and the HF_TOKEN secret below.
RUN_SMOKE = False
if not RUN_SMOKE:
    print("Safe setup only. Enable GPU + Internet, select HF_TOKEN under Add-ons → Secrets, then set RUN_SMOKE=True and run this cell.")
else:
    import hashlib, json, subprocess, sys
    from pathlib import Path

    packages = [
        "transformers==5.22.1",
        "peft==0.21.2",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
    smoke_source = '#!/usr/bin/env python3\n"""Run one synthetic 8B NF4 LoRA optimizer step on a Kaggle GPU.\n\nThis checks runtime compatibility only. It does not read the research corpus,\nproduce a model result, or certify that an experiment dataset is ready.\n"""\nfrom __future__ import annotations\n\nimport json\nimport hashlib\nimport os\nimport sys\nimport time\nfrom importlib.metadata import version\nfrom pathlib import Path\n\n\nMODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"\nEXPECTED = {\n    "transformers": "5.22.1",\n    "peft": "0.21.2",\n    "accelerate": "1.15.0",\n    "bitsandbytes": "0.50.2",\n}\n\n\ndef get_token() -> str:\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if token:\n        return token\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        token = UserSecretsClient().get_secret("HF_TOKEN")\n    except Exception as exc:  # Kaggle-only secret service; never print credential.\n        raise RuntimeError(f"HF_TOKEN unavailable ({type(exc).__name__})") from exc\n    if not token:\n        raise RuntimeError("HF_TOKEN is empty")\n    return token\n\n\ndef main() -> int:\n    output = Path(os.environ.get("R1_SMOKE_OUTPUT", "/kaggle/working/r1_8b_runtime_smoke.json"))\n    output.parent.mkdir(parents=True, exist_ok=True)\n    receipt: dict = {\n        "status": "STARTED",\n        "model_id": MODEL_ID,\n        "scope": "SYNTHETIC_RUNTIME_SMOKE_ONLY",\n        "research_data_loaded": False,\n        "optimizer_steps_requested": 1,\n    }\n    output.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n\n    try:\n        os.environ.setdefault("HF_HOME", "/kaggle/working/.cache/huggingface-r1-8b-smoke")\n        os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n        import torch\n        from huggingface_hub import HfApi, hf_hub_download\n        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n        from transformers import (\n            AutoModelForCausalLM,\n            AutoTokenizer,\n            BitsAndBytesConfig,\n            Trainer,\n            TrainingArguments,\n        )\n\n        actual = {name: version(name) for name in EXPECTED}\n        receipt["package_versions"] = actual\n        if actual != EXPECTED:\n            raise RuntimeError(f"Pinned package mismatch: expected={EXPECTED}, actual={actual}")\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA GPU is not available")\n        torch.cuda.set_device(0)\n        receipt["gpus"] = [\n            {\n                "name": torch.cuda.get_device_name(i),\n                "memory_gib": round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),\n            }\n            for i in range(torch.cuda.device_count())\n        ]\n        token = get_token()\n        info = HfApi(token=token).model_info(MODEL_ID, token=token)\n        revision = info.sha\n        if not revision or len(revision) != 40:\n            raise RuntimeError("Could not resolve a full immutable Hugging Face commit")\n        receipt["model_revision"] = revision\n        config_path = Path(\n            hf_hub_download(\n                repo_id=MODEL_ID,\n                filename="config.json",\n                revision=revision,\n                token=token,\n            )\n        )\n        config = json.loads(config_path.read_text(encoding="utf-8"))\n        if not isinstance(config, dict) or config.get("model_type") != "llama":\n            raise RuntimeError("Pinned config is not a Llama model config")\n        receipt["config_sha256"] = hashlib.sha256(config_path.read_bytes()).hexdigest()\n        receipt["config_model_type"] = config["model_type"]\n        output.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n\n        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token, revision=revision)\n        quantization = BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=torch.float16,\n        )\n        started = time.monotonic()\n        model = AutoModelForCausalLM.from_pretrained(\n            MODEL_ID,\n            token=token,\n            revision=revision,\n            dtype=torch.float16,\n            quantization_config=quantization,\n            device_map={"": 0},\n            low_cpu_mem_usage=True,\n        )\n        receipt["model_load_seconds"] = round(time.monotonic() - started, 2)\n        receipt["model_loaded_4bit"] = bool(getattr(model, "is_loaded_in_4bit", False))\n        if not receipt["model_loaded_4bit"]:\n            raise RuntimeError("Model did not report a 4-bit load")\n\n        model.config.use_cache = False\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n        model = get_peft_model(\n            model,\n            LoraConfig(\n                r=16,\n                lora_alpha=32,\n                lora_dropout=0.05,\n                bias="none",\n                task_type="CAUSAL_LM",\n                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n            ),\n        )\n        messages = [\n            {"role": "user", "content": "For this software check only, reply with exactly OK."},\n            {"role": "assistant", "content": "OK"},\n        ]\n        prompt_ids = tokenizer.apply_chat_template(\n            messages[:-1], tokenize=True, add_generation_prompt=True\n        )\n        full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)\n        if full_ids[: len(prompt_ids)] != prompt_ids or len(full_ids) <= len(prompt_ids):\n            raise RuntimeError("Assistant-only loss mask does not match the tokenizer chat template")\n        sample = {\n            "input_ids": full_ids,\n            "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids) :],\n        }\n\n        def collate(batch: list[dict]) -> dict[str, torch.Tensor]:\n            width = max(len(row["input_ids"]) for row in batch)\n            input_ids, labels, masks = [], [], []\n            for row in batch:\n                pad = width - len(row["input_ids"])\n                input_ids.append(row["input_ids"] + [tokenizer.pad_token_id or tokenizer.eos_token_id] * pad)\n                labels.append(row["labels"] + [-100] * pad)\n                masks.append([1] * len(row["input_ids"]) + [0] * pad)\n            return {\n                "input_ids": torch.tensor(input_ids, dtype=torch.long),\n                "labels": torch.tensor(labels, dtype=torch.long),\n                "attention_mask": torch.tensor(masks, dtype=torch.long),\n            }\n\n        args = TrainingArguments(\n            output_dir="/kaggle/working/r1_8b_runtime_smoke_trainer",\n            max_steps=1,\n            per_device_train_batch_size=1,\n            gradient_accumulation_steps=1,\n            learning_rate=2e-4,\n            fp16=True,\n            gradient_checkpointing=True,\n            optim="adamw_torch",\n            logging_strategy="no",\n            save_strategy="no",\n            report_to="none",\n            dataloader_num_workers=0,\n            disable_tqdm=True,\n            seed=42,\n            data_seed=42,\n        )\n        trainer = Trainer(model=model, args=args, train_dataset=[sample], data_collator=collate)\n        trainer.train()\n        steps = int(trainer.state.global_step)\n        if steps != 1:\n            raise RuntimeError(f"Trainer completed {steps} optimizer steps, expected 1")\n        receipt.update(\n            {\n                "status": "PASS_RUNTIME_ONLY",\n                "trainer_global_step": steps,\n                "last_loss": float(trainer.state.log_history[-1]["loss"]),\n                "cuda_peak_memory_gib": round(torch.cuda.max_memory_reserved(0) / 1024**3, 2),\n                "research_data_loaded": False,\n                "interpretation": "Exact pinned stack loaded the gated 8B checkpoint as NF4 and completed one synthetic LoRA Trainer step; this is not an experiment result.",\n            }\n        )\n        del trainer, model, tokenizer\n        torch.cuda.empty_cache()\n    except Exception as exc:\n        receipt.update(\n            {\n                "status": "FAIL_RUNTIME_ONLY",\n                "error_type": type(exc).__name__,\n                "error": str(exc)[:2000],\n                "research_data_loaded": False,\n            }\n        )\n        output.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n        raise\n\n    output.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(receipt, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
    expected_sha256 = "014e8af05695ed3c5b50ed48be25df4848e5c8c845ed619f92f05e2ff44d869d"
    actual_sha256 = hashlib.sha256(smoke_source.encode("utf-8")).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Smoke runner hash mismatch: {actual_sha256}")
    runner = {"__name__": "r1_8b_runtime_smoke"}
    exec(compile(smoke_source, "kaggle_8b_runtime_smoke.py", "exec"), runner)
    exit_code = runner["main"]()
    receipt = json.loads(Path("/kaggle/working/r1_8b_runtime_smoke.json").read_text())
    if exit_code != 0 or receipt.get("status") != "PASS_RUNTIME_ONLY":
        raise RuntimeError(f"8B smoke did not pass: {receipt}")
    print(json.dumps(receipt, indent=2))
